In [ ]:
# Install required packages if needed
# !pip install torch torchvision pillow matplotlib pyyaml

In [ ]:
import torch
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
import torch.utils.data
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader
import yaml
import time

In [ ]:
# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Dataset Class

Reusing the same dataset class for YOLO format labels

In [ ]:
class SnowPoleDataset(torch.utils.data.Dataset):
    def __init__(self, data_root, image_folder, label_folder, transforms=None):
        """
        Args:
            data_root: Root directory containing data
            image_folder: Folder path relative to data_root for images
            label_folder: Folder path relative to data_root for labels
            transforms: Optional transforms to be applied
        """
        self.data_root = data_root
        self.image_dir = os.path.join(data_root, image_folder)
        self.label_dir = os.path.join(data_root, label_folder)
        self.transforms = transforms
        
        # Get all image files
        self.images = [f for f in os.listdir(self.image_dir) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.images.sort()
        
        print(f"Found {len(self.images)} images in {self.image_dir}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Load image
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        
        # Load corresponding label (YOLO format)
        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)
        
        boxes = []
        labels = []
        
        img_width, img_height = img.size
        
        # Parse YOLO format labels
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id, cx, cy, w, h = map(float, parts)
                        
                        # Convert from YOLO format to [xmin, ymin, xmax, ymax]
                        xmin = (cx - w / 2) * img_width
                        ymin = (cy - h / 2) * img_height
                        xmax = (cx + w / 2) * img_width
                        ymax = (cy + h / 2) * img_height
                        
                        boxes.append([xmin, ymin, xmax, ymax])
                        labels.append(int(class_id) + 1)  # +1 because 0 is background
        
        # Convert to tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Handle images with no boxes
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx])
        }
        
        if self.transforms:
            img = self.transforms(img)
        else:
            img = torchvision.transforms.ToTensor()(img)
        
        return img, target

## Load Dataset Configuration

In [ ]:
# Load data.yaml
with open('../../data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset configuration:")
print(data_config)

# Get data root and paths
data_root = os.path.abspath(os.path.join('../../', data_config['path']))
train_images = data_config['train']
val_images = data_config['val']
num_classes = data_config['nc'] + 1  # +1 for background

# Set subset size for testing
SUBSET_FRACTION = 1.0  # Use 100% of data

print(f"\nData root: {data_root}")
print(f"Number of classes (including background): {num_classes}")
print(f"Using {SUBSET_FRACTION * 100:.0f}% of the dataset")

In [ ]:
# Create datasets
train_dataset_full = SnowPoleDataset(
    data_root=data_root,
    image_folder=train_images,
    label_folder='labels/Train/train'
)

val_dataset_full = SnowPoleDataset(
    data_root=data_root,
    image_folder=val_images,
    label_folder='labels/Validation/val'
)

# Create subsets if needed
if SUBSET_FRACTION < 1.0:
    train_size = int(len(train_dataset_full) * SUBSET_FRACTION)
    val_size = int(len(val_dataset_full) * SUBSET_FRACTION)
    
    train_indices = torch.randperm(len(train_dataset_full))[:train_size].tolist()
    val_indices = torch.randperm(len(val_dataset_full))[:val_size].tolist()
    
    train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
    val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)
    
    print(f"Full training samples: {len(train_dataset_full)} -> Using: {len(train_dataset)}")
    print(f"Full validation samples: {len(val_dataset_full)} -> Using: {len(val_dataset)}")
else:
    train_dataset = train_dataset_full
    val_dataset = val_dataset_full
    print(f"Training samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")

## Create Lightweight Model

Using SSDLite320 with MobileNetV3 backbone - optimized for mobile/edge devices

In [ ]:
def get_lightweight_model(num_classes):
    """
    Create MobileNetV3-SSDLite model
    - Input size: 320x320 (smaller = faster)
    - Backbone: MobileNetV3-Large (efficient depthwise separable convolutions)
    - Head: SSDLite (lightweight version of SSD)
    """
    # Load pretrained MobileNetV3-SSDLite
    model = ssdlite320_mobilenet_v3_large(pretrained=True)
    
    # Get the number of input channels for the classification head
    in_channels = [672, 480, 512, 256, 256, 128]  # MobileNetV3 feature channels
    num_anchors = model.head.classification_head.num_anchors
    
    # Replace classification head for our number of classes
    model.head.classification_head = SSDLiteClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes
    )
    
    return model

# Create model
model = get_lightweight_model(num_classes)
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = total_params * 4 / (1024**2)  # float32

print(f"\nModel: MobileNetV3-SSDLite320")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Estimated model size: {model_size_mb:.2f} MB")
print(f"\nComparison with Faster R-CNN:")
print(f"  Size reduction: {160/model_size_mb:.1f}x smaller")
print(f"  Parameters reduction: {41000000/total_params:.1f}x fewer parameters")

## Data Loaders

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

# Increase batch size for lightweight model (can handle more)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,  # Larger batch size possible with lightweight model
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## Training Setup

In [ ]:
# Optimizer - Adam works well for lightweight models
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=0.001, weight_decay=0.0001)

# Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

num_epochs = 15  # Lightweight models may need slightly more epochs

## Training Loop

In [ ]:
# Training loop
train_losses = []
val_losses = []
epoch_times = []

for epoch in range(num_epochs):
    epoch_start = time.time()
    
    # Training phase
    model.train()
    epoch_loss = 0
    
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 30)
    
    for i, (images, targets) in enumerate(train_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        
        if (i + 1) % 10 == 0:
            print(f"Batch [{i+1}/{len(train_loader)}], Loss: {losses.item():.4f}")
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_losses.append(avg_val_loss)
    
    # Update learning rate based on validation loss
    lr_scheduler.step(avg_val_loss)
    
    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Time: {epoch_time:.2f}s")
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        os.makedirs('checkpoints', exist_ok=True)
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, f'checkpoints/mobilenet_ssd_epoch_{epoch+1}.pth')
        print(f"Checkpoint saved at epoch {epoch + 1}")

print("\nTraining completed!")
print(f"Average epoch time: {np.mean(epoch_times):.2f}s")

## Plot Training Progress

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, marker='o', label='Train Loss')
plt.plot(range(1, num_epochs + 1), val_losses, marker='s', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), epoch_times, marker='o', color='green')
plt.xlabel('Epoch')
plt.ylabel('Time (seconds)')
plt.title('Training Time per Epoch')
plt.grid(True)

plt.tight_layout()
plt.savefig('mobilenet_ssd_training.png')
plt.show()

## Evaluation Metrics

In [ ]:
# Reuse evaluation functions from Faster R-CNN notebook
from collections import defaultdict

def calculate_iou(box1, box2):
    """Calculate Intersection over Union between two boxes"""
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    if inter_x_max < inter_x_min or inter_y_max < inter_y_min:
        return 0.0
    
    inter_area = (inter_x_max - inter_x_min) * (inter_y_max - inter_y_min)
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

def calculate_precision_recall_ap(predictions, ground_truths, iou_threshold=0.5):
    predictions = sorted(predictions, key=lambda x: x[1], reverse=True)
    total_gt = sum(len(boxes) for boxes in ground_truths.values())
    
    if total_gt == 0:
        return 0.0, 0.0, 0.0
    
    gt_matched = {img_id: [False] * len(boxes) for img_id, boxes in ground_truths.items()}
    true_positives = []
    false_positives = []
    
    for img_id, conf, pred_box in predictions:
        if img_id not in ground_truths:
            false_positives.append(1)
            true_positives.append(0)
            continue
        
        gt_boxes = ground_truths[img_id]
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_matched[img_id][gt_idx]:
                continue
            iou = calculate_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx != -1:
            gt_matched[img_id][best_gt_idx] = True
            true_positives.append(1)
            false_positives.append(0)
        else:
            false_positives.append(1)
            true_positives.append(0)
    
    tp_cumsum = np.cumsum(true_positives)
    fp_cumsum = np.cumsum(false_positives)
    precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-10)
    recalls = tp_cumsum / total_gt
    
    ap = 0.0
    for recall_threshold in np.linspace(0, 1, 11):
        precisions_above_threshold = precisions[recalls >= recall_threshold]
        if len(precisions_above_threshold) > 0:
            ap += np.max(precisions_above_threshold)
    ap /= 11.0
    
    final_precision = precisions[-1] if len(precisions) > 0 else 0.0
    final_recall = recalls[-1] if len(recalls) > 0 else 0.0
    
    return final_precision, final_recall, ap

def evaluate_model(model, dataset, conf_threshold=0.25):
    model.eval()
    all_predictions = []
    ground_truths = {}
    inference_times = []
    
    print("Collecting predictions...")
    
    with torch.no_grad():
        for idx in range(len(dataset)):
            img, target = dataset[idx]
            
            # Measure inference time
            start_time = time.time()
            prediction = model([img.to(device)])[0]
            inference_time = time.time() - start_time
            inference_times.append(inference_time)
            
            keep = prediction['scores'] > conf_threshold
            pred_boxes = prediction['boxes'][keep].cpu().numpy()
            pred_scores = prediction['scores'][keep].cpu().numpy()
            
            for box, score in zip(pred_boxes, pred_scores):
                all_predictions.append((idx, score, box))
            
            gt_boxes = target['boxes'].numpy()
            ground_truths[idx] = gt_boxes
            
            if (idx + 1) % 50 == 0:
                print(f"Processed {idx + 1}/{len(dataset)} images")
    
    # Calculate metrics
    precision_50, recall_50, ap_50 = calculate_precision_recall_ap(
        all_predictions, ground_truths, iou_threshold=0.5
    )
    
    aps = []
    for iou_thresh in np.arange(0.5, 1.0, 0.05):
        _, _, ap = calculate_precision_recall_ap(
            all_predictions, ground_truths, iou_threshold=iou_thresh
        )
        aps.append(ap)
    
    map_50_95 = np.mean(aps)
    avg_inference_time = np.mean(inference_times)
    fps = 1.0 / avg_inference_time
    
    results = {
        'Precision': precision_50,
        'Recall': recall_50,
        'mAP@50': ap_50,
        'mAP@0.5:0.95': map_50_95,
        'avg_inference_time_ms': avg_inference_time * 1000,
        'fps': fps
    }
    
    return results

In [ ]:
# Evaluate on validation set
print("\n" + "="*50)
print("EVALUATING ON VALIDATION SET")
print("="*50 + "\n")

val_results = evaluate_model(model, val_dataset, conf_threshold=0.25)

print("\n" + "="*50)
print("VALIDATION SET RESULTS")
print("="*50)
print(f"Precision:       {val_results['Precision']:.4f}")
print(f"Recall:          {val_results['Recall']:.4f}")
print(f"mAP@50:          {val_results['mAP@50']:.4f}")
print(f"mAP@0.5:0.95:    {val_results['mAP@0.5:0.95']:.4f}")
print(f"\nInference Speed:")
print(f"Avg time:        {val_results['avg_inference_time_ms']:.2f} ms")
print(f"FPS:             {val_results['fps']:.1f}")
print("\nEdge Device Suitability: ✓ EXCELLENT")
print(f"  - {'Fast' if val_results['fps'] > 30 else 'Moderate'} inference speed")
print(f"  - Lightweight architecture (~{model_size_mb:.1f} MB)")
print(f"  - Low memory footprint")

## Visualize Predictions

In [ ]:
def predict_and_visualize(model, dataset, idx, conf_threshold=0.5):
    model.eval()
    img, target = dataset[idx]
    
    with torch.no_grad():
        prediction = model([img.to(device)])[0]
    
    keep = prediction['scores'] > conf_threshold
    boxes = prediction['boxes'][keep].cpu().numpy()
    scores = prediction['scores'][keep].cpu().numpy()
    gt_boxes = target['boxes'].numpy()
    
    img_np = img.permute(1, 2, 0).numpy()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    # Ground truth
    ax1.imshow(img_np)
    for box in gt_boxes:
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor='g', facecolor='none'
        )
        ax1.add_patch(rect)
    ax1.set_title(f'Ground Truth ({len(gt_boxes)} poles)')
    ax1.axis('off')
    
    # Predictions
    ax2.imshow(img_np)
    for box, score in zip(boxes, scores):
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=2, edgecolor='r', facecolor='none'
        )
        ax2.add_patch(rect)
        ax2.text(xmin, ymin - 5, f'{score:.2f}', 
                color='red', fontsize=10, weight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    ax2.set_title(f'MobileNetV3-SSD ({len(boxes)} poles, conf>{conf_threshold})')
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return boxes, scores

# Test on validation samples
for i in range(min(5, len(val_dataset))):
    boxes, scores = predict_and_visualize(model, val_dataset, i, conf_threshold=0.5)

## Save Results

In [ ]:
import json

# Save comprehensive metrics
metrics_dict = {
    'model': 'MobileNetV3-SSDLite320',
    'model_size_mb': float(model_size_mb),
    'total_parameters': int(total_params),
    'dataset_fraction': SUBSET_FRACTION,
    'num_epochs': num_epochs,
    'validation_samples': len(val_dataset),
    'metrics': {
        'Precision': float(val_results['Precision']),
        'Recall': float(val_results['Recall']),
        'mAP@50': float(val_results['mAP@50']),
        'mAP@0.5:0.95': float(val_results['mAP@0.5:0.95'])
    },
    'performance': {
        'avg_inference_time_ms': float(val_results['avg_inference_time_ms']),
        'fps': float(val_results['fps'])
    },
    'final_train_loss': float(train_losses[-1]),
    'final_val_loss': float(val_losses[-1]),
    'edge_device_ready': True
}

with open('mobilenet_ssd_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)

print("Metrics saved to 'mobilenet_ssd_metrics.json'")
print("\nFull Metrics Summary:")
print(json.dumps(metrics_dict, indent=2))

## Save Final Model

In [ ]:
# Save the final model
torch.save(model.state_dict(), 'snow_pole_mobilenet_ssd_final.pth')
print("Final model saved as 'snow_pole_mobilenet_ssd_final.pth'")

# Also save as TorchScript for deployment
model.eval()
example_input = torch.rand(1, 3, 320, 320).to(device)
traced_model = torch.jit.trace(model, example_input)
traced_model.save('snow_pole_mobilenet_ssd_traced.pt')
print("TorchScript model saved as 'snow_pole_mobilenet_ssd_traced.pt'")
print("\nThis TorchScript model can be deployed on edge devices without Python!")

## Model Comparison Summary

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON: Faster R-CNN vs MobileNetV3-SSD")
print("="*60)

comparison = {
    'Metric': ['Model Size', 'Parameters', 'Inference Speed', 'Edge Device Ready'],
    'Faster R-CNN': ['~160 MB', '~41M', '~5-15 FPS', 'No ❌'],
    'MobileNetV3-SSD': [f'{model_size_mb:.1f} MB', f'{total_params/1e6:.1f}M', 
                        f'{val_results["fps"]:.1f} FPS', 'Yes ✓']
}

import pandas as pd
df = pd.DataFrame(comparison)
print(df.to_string(index=False))

print("\n" + "="*60)
print("RECOMMENDATION FOR EDGE DEPLOYMENT:")
print("="*60)
print("✓ MobileNetV3-SSD is HIGHLY SUITABLE for edge devices")
print(f"  - {160/model_size_mb:.0f}x smaller model size")
print(f"  - {val_results['fps']/10:.0f}x faster inference (estimated)")
print(f"  - Compatible with mobile CPUs, Raspberry Pi, Jetson Nano")
print(f"  - Can be quantized further for even better performance")
print("\nNext steps for deployment:")
print("  1. Quantize model (INT8) for 4x size reduction")
print("  2. Use ONNX format for cross-platform deployment")
print("  3. Deploy with TensorRT for NVIDIA edge devices")
print("  4. Use PyTorch Mobile for Android/iOS")